<a id="top"></a>

# Lab L2204: author a skill and wire it into the loader

```yaml
title: "Lab L2204: author a skill and wire it into the loader"
keywords: skill, author, description, trigger, jit loader, load_skill, create_agent, catalog, token cost, silent miss, vague description
estimated duration: 35
```

> **Lesson:** L22. **Roadmap:** [objectives.md](../../../../docs/origin/lesson_roadmaps/L22/objectives.md).

**You will be able to:**

- Author a markdown skill whose **description** states *when* it applies.
- Wire that skill into a just-in-time loader and measure its context cost.
- Show that a **vague description** causes a *silent miss* -- the skill never loads.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 - Author a skill](#2-problem-1---author-a-skill)
- [3. Problem 2 - Register it and measure the cost](#3-problem-2---register-it-and-measure-the-cost)
- [4. Problem 3 - Run a triggering task](#4-problem-3---run-a-triggering-task)
- [5. Problem 4 - The vague description silently misses](#5-problem-4---the-vague-description-silently-misses)
- [6. Problem 5 - Skill or system prompt? (written)](#6-problem-5---skill-or-system-prompt-written)

## 1. Setup

Everything here is **given**: the offline `FakeModel`, the `Skill` type, and the just-in-time loader (`skill_catalog`, `run_skill_loader`) from the demo notebook -- the L11 `create_agent` shallow agent whose one tool is `load_skill`. A starter shelf holds two skills. Run this cell as-is -- no edits, no API key.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Annotated

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage

from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)


def rough_tokens(text: str) -> int:
    """~4 characters per token -- enough to COMPARE context sizes."""
    return len(text) // 4


@dataclass(frozen=True)
class Skill:
    """A name, a description that says WHEN it applies, and a body of instructions."""

    name: str
    description: str
    body: str


def skill_catalog(skills: dict[str, Skill]) -> str:
    """The always-on view: name + description for every skill."""
    return "\n".join(f"- {s.name}: {s.description}" for s in skills.values())


@dataclass
class JitResult:
    final_text: str
    loaded: list[str]
    context_tokens: list[int]


def context_before_each_turn(system: str, messages: list[BaseMessage]) -> list[int]:
    """Context size (tokens) the model saw going into each of its turns, read off
    the run's returned messages. The always-on baseline is the system prompt (the
    catalog); the jump between turns is a skill body arriving just in time."""
    sizes: list[int] = []
    seen = system
    for msg in messages:
        if isinstance(msg, AIMessage):
            sizes.append(rough_tokens(seen))
        seen += "\n" + (msg.content if isinstance(msg.content, str) else str(msg.content))
    return sizes


async def run_skill_loader(model: FakeModel, skills: dict[str, Skill], user_msg: str) -> JitResult:
    """The L11 `create_agent` shallow agent whose one tool reads a skill body into
    context on demand, with the catalog as its system prompt. The loop is a
    `create_agent` freebie -- only the load_skill tool + catalog are new."""

    def load_skill(
        name: Annotated[str, "the exact skill name from the catalog to read into context"],
    ) -> str:
        """Read a skill's full body into context, or return an error string."""
        skill = skills.get(name)
        return skill.body if skill is not None else f"no such skill: {name!r}"

    system = (
        "You are a support agent. Skills are listed as `name: description`. "
        "Call load_skill(name) when a skill matches, then follow it.\n\n"
        f"Available skills:\n{skill_catalog(skills)}"
    )
    agent = create_agent(model, [load_skill], system_prompt=system)
    messages = (await agent.ainvoke({"messages": [HumanMessage(user_msg)]}))["messages"]
    loaded = [
        call["args"]["name"]
        for msg in messages
        if isinstance(msg, AIMessage)
        for call in msg.tool_calls
        if call["name"] == "load_skill"
    ]
    return JitResult(
        final_text=str(messages[-1].content),
        loaded=loaded,
        context_tokens=context_before_each_turn(system, messages),
    )


# A starter shelf with two skills already on it.
SKILLS: dict[str, Skill] = {
    "refund-policy": Skill(
        "refund-policy",
        "Apply when the user asks to process a refund, return, or money-back request.",
        "# Refund policy\n1. Confirm within the 30-day window.\n2. 10% restocking on opened items.\n",
    ),
    "shipping-eta": Skill(
        "shipping-eta",
        "Apply when the user asks where their order is or when it will arrive.",
        "# Shipping ETA\n1. Standard is 5-7 business days; express is 2.\n",
    ),
}
print("starter shelf:", list(SKILLS))

[↑ Back to top](#top)

## 2. Problem 1 - Author a skill

Author an **`escalation`** skill: what an agent should do when a customer is angry or asks for a manager. The **description** is the load-bearing part -- write it so it fires on complaints and *not* on ordinary questions. The **body** is 2-4 imperative steps *for the agent to follow*.

In [ ]:
# TODO: fill in a discriminating description and an imperative body.
escalation = Skill(
    name="escalation",
    description="",  # TODO: state WHEN this applies (angry / complaint / "get me a manager")
    body="",  # TODO: 2-4 imperative steps for the agent
)
print(escalation.description)

[↑ Back to top](#top)

## 3. Problem 2 - Register it and measure the cost

Add `escalation` to the shelf, then compare two worlds: **all bodies always-on** (as if they sat in the system prompt) vs. **JIT** (catalog always, plus only the body actually loaded). Use `rough_tokens` and `skill_catalog`.

In [ ]:
SKILLS["escalation"] = escalation  # register it

# TODO: compute and print three numbers:
#   - catalog_tokens: rough_tokens of the catalog (always-on, cheap)
#   - all bodies always-on: catalog_tokens + sum of every body's tokens
#   - escalation JIT: catalog_tokens + just the escalation body's tokens

[↑ Back to top](#top)

## 4. Problem 3 - Run a triggering task

Below is a **scripted** model (given) that loads your `escalation` skill, then answers. Run it through `run_skill_loader` on a task that *should* trigger escalation, and print which skills loaded and the per-turn context sizes. Notice the second number jump: that is the body arriving **just in time**.

In [ ]:
# Given: a scripted model that loads escalation, then answers.
angry_model = FakeModel(
    [
        tool_reply(tool_call("s1", "load_skill", {"name": "escalation"})),
        text_reply(
            "I am sorry about this. I have escalated you to a manager who will call within 2 hours."
        ),
    ]
)

# TODO: `await run_skill_loader(...)` on angry_model with an angry / manager-request task, then
#       print result.loaded, result.context_tokens, and result.final_text.

[↑ Back to top](#top)

## 5. Problem 4 - The vague description silently misses

Here is the **same** skill with a vague description -- `"helps with customer stuff"` -- and a model that, faced with that fuzzy catalog, never loads it. Build a shelf using the vague variant, run the same angry task, and confirm `loaded` is **empty**. The capability existed; the description failed to fire it.

In [ ]:
# Given: a vague-description copy, and a model that never loads the skill.
vague_escalation = Skill(
    name="escalation",
    description="helps with customer stuff",
    body=escalation.body,
)
missed_model = FakeModel([text_reply("Okay, noted. Is there anything else?")])

# TODO: make a shelf where 'escalation' has the VAGUE description, `await run_skill_loader(...)`
#       with missed_model on the angry task, and print result.loaded (expect []) and
#       result.final_text.

[↑ Back to top](#top)

## 6. Problem 5 - Skill or system prompt? (written)

_Double-click to edit._

**Q1.** In one sentence, why did the vague description cause a *silent miss*?

**Q2.** Should `escalation` be a **skill** or **system-prompt content**? Defend your call using the always-on cost you measured in Problem 2.

_TODO: your answers here._

[↑ Back to top](#top)